# PCF Sweep

Launch a Polynomial Continued Fraction sweep and visualise hits.

In [ ]:
from zeta_hunter.pcf.families import APERY, ZAGIER, RAMANUJAN
from zeta_hunter.pcf.sweeper import PCFSweeper
from zeta_hunter.logger import RunLogger
from dataclasses import replace
from datetime import datetime

FAMILY = ZAGIER                          # APERY | ZAGIER | RAMANUJAN
COEFF_RANGE = range(-10, 11)             # start small; scale up for real runs
RUN_ID = f"sweep_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

family = replace(FAMILY, coeff_range=COEFF_RANGE)
print(f"Family:       {family.name}")
print(f"Combinations: {family.total_combinations:,}")
print(f"Run ID:       {RUN_ID}")

In [ ]:
logger = RunLogger(RUN_ID)
sweeper = PCFSweeper(family, logger)
sweeper.sweep()
print("Sweep complete.")


In [ ]:
import json, pathlib
data = json.loads(pathlib.Path(f"runs/{RUN_ID}.json").read_text())
stats = data["stats"]
print(f"Scanned:    {stats['total_scanned']:,}")
print(f"Stage 1:    {stats['stage1_hits']} hits")
print(f"Stage 2:    {stats['stage2_hits']} hits")
print(f"Elapsed:    {stats['elapsed_seconds']:.1f}s")
print(f"Throughput: {stats['throughput_per_hour']:,} PCFs/hour")

In [ ]:
import matplotlib.pyplot as plt, numpy as np
stage1 = data["stage1_hits"]
if stage1:
    errors = [h["error"] for h in stage1]
    plt.figure(figsize=(8, 4))
    plt.hist(np.log10(errors), bins=30, color="steelblue", edgecolor="black")
    plt.xlabel("log10(|CF - target|)")
    plt.ylabel("Count")
    plt.title(f"Stage 1 hit distribution — {family.name} family")
    plt.tight_layout()
    plt.show()
else:
    print("No Stage 1 hits in this sweep.")